# L1 — DynamoDB Orders Table Setup

## What You'll Learn

- How to create a DynamoDB table with a composite key and multiple Global Secondary Indexes (GSIs)
- How to load structured order data from a JSON file into DynamoDB
- How to query the table using the primary key and GSIs to support different access patterns
- How to publish resource IDs to SSM Parameter Store for downstream agents

## Overview

AnyCompany's Order Agent needs to look up customer orders in real time — by order ID, by customer, by status, and by product category. This notebook creates the `CustomerOrders` DynamoDB table that powers those lookups.

We create the table with a composite key (`customerId` + `orderId`) and four Global Secondary Indexes (GSIs) to support every access pattern the Order Agent needs. We then load 25 sample orders with embedded product details (name, category, price, image URL) and verify the data with a few representative queries. The table name is stored in SSM Parameter Store so the Order Agent in L3 can retrieve it at runtime without hardcoding.

## Architecture

```
  customer_orders_data.json
          │
          ▼
  ┌───────────────────────────────────────────────────┐
  │           DynamoDB: CustomerOrders                │
  │                                                   │
  │  Primary Key: customerId (HASH) + orderId (RANGE) │
  │                                                   │
  │  GSIs:                                            │
  │  ├── OrderIdIndex      → look up by order ID      │
  │  ├── StatusIndex       → filter by status         │
  │  ├── CategoryIndex     → filter by category       │
  │  └── ProductNameIndex  → filter by product name   │
  └───────────────────────────────────────────────────┘
          │
          ▼
  L3 Order Agent
  (check_order_details Lambda → dynamodb.query)
```

## Steps

1. Create the `CustomerOrders` DynamoDB table with GSIs
2. Load 25 sample orders from `customer_orders_data.json`
3. Verify the data with sample queries
4. Store the table name in SSM Parameter Store

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

Install required packages.

In [ ]:
%pip install boto3==1.40.50 --quiet

Import libraries and resolve the AWS account ID and region.

In [ ]:
import boto3
import json
import time
from decimal import Decimal
from boto3.dynamodb.types import TypeSerializer

sts = boto3.client("sts")
identity = sts.get_caller_identity()
ACCOUNT_ID = identity["Account"]
REGION = boto3.session.Session().region_name or "us-east-1"

dynamodb = boto3.client("dynamodb", region_name=REGION)

print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

## Step 1: Create DynamoDB Table

Create the `CustomerOrders` table from the JSON definition in `customer_orders_table.json`. Waits for the table to reach ACTIVE status before continuing.

In [ ]:
def create_table(table_def):
    """Create a DynamoDB table from a JSON definition."""
    table_name = table_def["TableName"]
    params = {
        "TableName": table_name,
        "BillingMode": table_def["BillingMode"],
        "AttributeDefinitions": table_def["AttributeDefinitions"],
        "KeySchema": table_def["KeySchema"],
    }
    if "GlobalSecondaryIndexes" in table_def:
        params["GlobalSecondaryIndexes"] = table_def["GlobalSecondaryIndexes"]
    if "Tags" in table_def:
        params["Tags"] = table_def["Tags"]

    try:
        dynamodb.create_table(**params)
        print(f"Creating table: {table_name}...")
        waiter = dynamodb.get_waiter("table_exists")
        waiter.wait(TableName=table_name)
        print(f"  Table {table_name} is ACTIVE.")
    except dynamodb.exceptions.ResourceInUseException:
        print(f"  Table {table_name} already exists.")

    return table_name


with open("./sample-data/customer_orders_table.json") as f:
    table_def = json.load(f)

TABLE_NAME = create_table(table_def)

## Step 2: Load Sample Data

Load 25 sample orders from `customer_orders_data.json` into the table. Each item includes order details and embedded product information (name, category, price, image URL).

In [ ]:
serializer = TypeSerializer()


def convert_floats_to_decimal(obj):
    if isinstance(obj, float):
        return Decimal(str(obj))
    elif isinstance(obj, dict):
        return {k: convert_floats_to_decimal(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_floats_to_decimal(i) for i in obj]
    return obj


def serialize_item(item):
    item = convert_floats_to_decimal(item)
    return {k: serializer.serialize(v) for k, v in item.items()}


# Load order data
with open("./sample-data/customer_orders_data.json") as f:
    orders = json.load(f)

print(f"Loading {len(orders)} orders into {TABLE_NAME}...")
for order in orders:
    clean_order = {k: v for k, v in order.items() if v is not None}
    dynamodb.put_item(TableName=TABLE_NAME, Item=serialize_item(clean_order))
    print(f"  Added: {order['orderId']} (customer: {order['customerId']}, product: {order['productName']}, status: {order['status']})")

print(f"\nLoaded {len(orders)} orders into {TABLE_NAME}.")

## Step 3: Verify Data

Verify the loaded data with three sample queries: total item count, orders by customer (primary key), order by ID (GSI), and orders by category (GSI) — the same access patterns the Order Agent uses.

In [ ]:
from boto3.dynamodb.types import TypeDeserializer

deserializer = TypeDeserializer()


def deserialize_item(item):
    return {k: deserializer.deserialize(v) for k, v in item.items()}


# Verify total count
scan_result = dynamodb.scan(TableName=TABLE_NAME, Select="COUNT")
print(f"{TABLE_NAME}: {scan_result['Count']} total items")

# Sample order lookup by customer
print("\n--- Orders for CUST-789 ---")
resp = dynamodb.query(
    TableName=TABLE_NAME,
    KeyConditionExpression="customerId = :cid",
    ExpressionAttributeValues={":cid": {"S": "CUST-789"}},
)
for item in resp["Items"][:3]:
    order = deserialize_item(item)
    print(f"  {order['orderId']}: {order['productName']} ({order['category']}) - ${order['totalAmount']} - {order['status']}")

# Sample order lookup by orderId (GSI)
print("\n--- Order Lookup (ORD-10001) ---")
resp = dynamodb.query(
    TableName=TABLE_NAME,
    IndexName="OrderIdIndex",
    KeyConditionExpression="orderId = :oid",
    ExpressionAttributeValues={":oid": {"S": "ORD-10001"}},
)
for item in resp["Items"]:
    order = deserialize_item(item)
    print(json.dumps(order, indent=2, default=str))

# Sample category lookup (GSI)
print("\n--- Electronics Orders ---")
resp = dynamodb.query(
    TableName=TABLE_NAME,
    IndexName="CategoryIndex",
    KeyConditionExpression="category = :cat",
    ExpressionAttributeValues={":cat": {"S": "Electronics"}},
)
for item in resp["Items"][:3]:
    order = deserialize_item(item)
    print(f"  {order['orderId']}: {order['productName']} - ${order['totalAmount']}")

## Step 4: Store Resource IDs in SSM Parameter Store

Publish the table name to SSM Parameter Store so the Order Agent in L3 can look it up at runtime without hardcoding.

In [ ]:
SSM_PREFIX = "/anycompany/agentcore"
ssm = boto3.client("ssm", region_name=REGION)

params = {
    f"{SSM_PREFIX}/dynamodb_orders_table": TABLE_NAME,
}

for name, value in params.items():
    ssm.put_parameter(Name=name, Value=value, Type="String", Overwrite=True)
    print(f"Stored: {name} = {value}")

print("\nAll resource IDs stored in SSM Parameter Store.")

## Summary

| Resource | Value |
|----------|-------|
| Table | `CustomerOrders` |
| Key Schema | `customerId` (HASH), `orderId` (RANGE) |
| GSIs | `OrderIdIndex`, `StatusIndex`, `CategoryIndex`, `ProductNameIndex` |
| Orders Loaded | 25 (with embedded product details) |


---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- DynamoDB table (`CustomerOrders`) and all GSIs
- SSM parameter

Uncomment and run this cell to delete all resources created in this notebook. Only run after completing all other notebooks.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.

# import boto3, os
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
# TABLE_NAME = "CustomerOrders"
#
# dynamodb = boto3.client("dynamodb", region_name=REGION)
# ssm = boto3.client("ssm", region_name=REGION)
#
# # 1. Delete DynamoDB table (GSIs are deleted automatically with the table)
# try:
#     dynamodb.delete_table(TableName=TABLE_NAME)
#     print(f"Deleting table: {TABLE_NAME}...")
#     waiter = dynamodb.get_waiter("table_not_exists")
#     waiter.wait(TableName=TABLE_NAME)
#     print(f"Table {TABLE_NAME} deleted successfully.")
# except dynamodb.exceptions.ResourceNotFoundException:
#     print(f"Table {TABLE_NAME} does not exist, skipping.")
# except Exception as e:
#     print(f"Table cleanup error: {e}")
#
# # 2. Delete SSM parameter
# try:
#     ssm.delete_parameter(Name=f"{SSM_PREFIX}/dynamodb_orders_table")
#     print(f"Deleted SSM parameter: {SSM_PREFIX}/dynamodb_orders_table")
# except ssm.exceptions.ParameterNotFound:
#     print("SSM parameter not found, skipping.")
# except Exception as e:
#     print(f"SSM cleanup error: {e}")
#
# print("\n✅ All Notebook 2 resources cleaned up.")